<a href="https://colab.research.google.com/github/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# STEP 0 — Project setup, file discovery, workbook audit
# PDF-driven workflow for Aleurodicus rugioperculatus analysis
# ============================================================

# ---------- 0.1 Mount Google Drive ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- 0.2 Imports ----------
import os
import re
import glob
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ---------- 0.3 User project paths ----------
PROJECT_ROOT = Path("/content/drive/MyDrive/Data Analysis March 2026")
OUTPUT_DIR = PROJECT_ROOT / "Mishu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_XLSX = PROJECT_ROOT / "Aleurodicus_rugioperculatus.xlsx"
TARGET_AUDIT_XLSX = OUTPUT_DIR / "Step0_Workbook_Audit.xlsx"
TARGET_META_JSON = OUTPUT_DIR / "Step0_Project_Metadata.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")

# ---------- 0.4 Helper: search files robustly ----------
def find_candidate_files(
    exact_name=None,
    fuzzy_patterns=None,
    search_roots=None,
    max_hits=300
):
    if search_roots is None:
        search_roots = [
            "/content",
            "/content/drive/MyDrive",
            "/mnt/data",
        ]

    candidates = []

    if exact_name:
        for root in search_roots:
            pattern = os.path.join(root, "**", exact_name)
            candidates.extend(glob.glob(pattern, recursive=True))

    if fuzzy_patterns:
        for fp in fuzzy_patterns:
            for root in search_roots:
                pattern = os.path.join(root, "**", fp)
                candidates.extend(glob.glob(pattern, recursive=True))

    seen = set()
    clean = []
    for p in candidates:
        if p not in seen and os.path.isfile(p):
            seen.add(p)
            clean.append(p)

    return clean[:max_hits]

# ---------- 0.5 Locate workbook ----------
xlsx_candidates = find_candidate_files(
    exact_name="Aleurodicus_rugioperculatus.xlsx",
    fuzzy_patterns=[
        "*Aleurodicus*.xlsx",
        "*rugioperculatus*.xlsx",
        "*.xlsx",
    ]
)

if not xlsx_candidates:
    raise FileNotFoundError(
        "No Excel workbook was found.\n\n"
        "Upload the workbook to Colab or place it in Drive, then rerun Step 0."
    )

print("\nCandidate Excel files found:")
for p in xlsx_candidates[:20]:
    print(" -", p)

chosen_xlsx = None
for p in xlsx_candidates:
    if os.path.basename(p) == "Aleurodicus_rugioperculatus.xlsx":
        chosen_xlsx = p
        break

if chosen_xlsx is None:
    chosen_xlsx = xlsx_candidates[0]

print(f"\nUsing workbook:\n{chosen_xlsx}")

if Path(chosen_xlsx).resolve() != TARGET_XLSX.resolve():
    shutil.copy2(chosen_xlsx, TARGET_XLSX)
    print(f"Workbook copied to:\n{TARGET_XLSX}")
else:
    print("Workbook already present in the project folder.")

# ---------- 0.6 Locate GitHub notebook URL text file (robust) ----------
def extract_github_ipynb_url(text):
    if not isinstance(text, str):
        return None
    text = text.strip()
    pattern = r"https?://github\.com/[^\s]+/blob/[^\s]+\.ipynb"
    match = re.search(pattern, text)
    return match.group(0) if match else None

github_url = None
github_source_file = None

txt_candidates = find_candidate_files(
    exact_name="Github-Link.txt",
    fuzzy_patterns=[
        "*Github*.txt",
        "*GitHub*.txt",
        "*github*.txt",
        "*.txt"
    ]
)

print("\nCandidate text files checked for notebook URL:")
for p in txt_candidates[:20]:
    print(" -", p)

# first pass: preferred filenames
preferred_txt = [p for p in txt_candidates if os.path.basename(p).lower() in ["github-link.txt", "github link.txt"]]
other_txt = [p for p in txt_candidates if p not in preferred_txt]

for candidate_list in [preferred_txt, other_txt]:
    for candidate in candidate_list:
        try:
            with open(candidate, "r", encoding="utf-8", errors="ignore") as f:
                txt = f.read()
            found_url = extract_github_ipynb_url(txt)
            if found_url:
                github_url = found_url
                github_source_file = candidate
                break
        except Exception:
            pass
    if github_url:
        break

# hard fallback so metadata JSON is never blank
if github_url is None:
    github_url = "https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb"
    github_source_file = "hardcoded_fallback"

print("\nGitHub notebook reference:")
print(github_url)
print("GitHub URL source:")
print(github_source_file)

# ---------- 0.7 Lock the PDF-based analysis template ----------
ANALYSIS_TEMPLATE = {
    "primary_reference_template": "Uploaded PDF statistical style",
    "ignore_for_statistical_decisions": "Methodology.docx ANOVA/LSD workflow",
    "time_scale_in_actual_workbook": ["24 HAS", "48 HAS", "72 HAS"],
    "baseline_column_prefix": "BS_",
    "life_stages": ["Egg", "Nymph", "Adult"],
    "mortality_correction": "Abbott corrected mortality",
    "assumption_tests": ["Shapiro-Wilk", "Levene"],
    "group_comparison": "Kruskal-Wallis + Conover-Iman post-hoc + Holm-Bonferroni adjustment",
    "factorial_nonparametric_model": "ART ANOVA",
    "dose_response_timepoint": "72 HAS",
    "dose_response_model": "4-parameter log-logistic (LL.4), fallback LL.3 if lower asymptote unstable",
    "multivariate_analysis": "PCA + KMO at 72 HAS",
    "table_style": "Mean ± SE with grouping letters",
    "figure_exports": ["PNG", "PDF"],
    "dpi": 300,
    "output_excel": "Final_Analysis_Mishu.xlsx"
}

print("\nLocked analysis template:")
for k, v in ANALYSIS_TEMPLATE.items():
    print(f" - {k}: {v}")

# ---------- 0.8 Read workbook ----------
xlsx = pd.ExcelFile(TARGET_XLSX)
sheet_names = xlsx.sheet_names

print("\nSheets found in workbook:")
for s in sheet_names:
    print(" -", s)

# ---------- 0.9 Expected structure ----------
required_columns = [
    "Conc", "Rep",
    "BS_Egg", "24_Egg", "48_Egg", "72_Egg",
    "BS_Nymph", "24_Nymph", "48_Nymph", "72_Nymph",
    "BS_Adult", "24_Adult", "48_Adult", "72_Adult"
]

# ---------- 0.10 Helpers for cleaning ----------
def normalize_sheet_name(sheet_name: str) -> str:
    s = str(sheet_name).strip()
    s = re.sub(r"\s+", " ", s)
    if s == "Tundra 20SP":
        s = "Tundra 20 SP"
    if s == "Ravzoom 14.5 SC":
        s = "Rav-zoom 14.5 SC"
    return s

def extract_concentration_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "control":
        return 0.0
    s = s.replace("ppm", "").strip()
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else np.nan

def classify_treatment_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    return "control" if s == "control" else "treated"

# ---------- 0.11 Audit each sheet ----------
audit_rows = []
cleaned_preview = {}

for raw_sheet_name in sheet_names:
    df = pd.read_excel(TARGET_XLSX, sheet_name=raw_sheet_name)

    missing_required = [c for c in required_columns if c not in df.columns]

    conc_missing_before = int(df["Conc"].isna().sum()) if "Conc" in df.columns else np.nan

    if "Conc" in df.columns:
        df["Conc"] = df["Conc"].ffill()
        df["Conc"] = df["Conc"].astype(str).str.strip()
        df["Conc"] = df["Conc"].replace({"nan": np.nan})

    conc_missing_after = int(df["Conc"].isna().sum()) if "Conc" in df.columns else np.nan

    if "Conc" in df.columns:
        df["Treatment_Type"] = df["Conc"].apply(classify_treatment_label)
        df["Conc_ppm"] = df["Conc"].apply(extract_concentration_numeric)

    normalized_name = normalize_sheet_name(raw_sheet_name)

    if "Conc" in df.columns:
        control_mask = df["Treatment_Type"].eq("control")
        control_reps = int(control_mask.sum())
        treated_levels = int(df.loc[~control_mask, "Conc_ppm"].dropna().nunique())
        treated_conc_list = sorted(df.loc[~control_mask, "Conc_ppm"].dropna().unique().tolist())
    else:
        control_reps = np.nan
        treated_levels = np.nan
        treated_conc_list = []

    numeric_cols = [c for c in df.columns if c not in ["Conc", "Rep", "Treatment_Type"]]
    numeric_missing = int(df[numeric_cols].isna().sum().sum()) if numeric_cols else 0

    audit_rows.append({
        "Raw_Sheet_Name": raw_sheet_name,
        "Standardized_Sheet_Name": normalized_name,
        "Rows": int(df.shape[0]),
        "Columns": int(df.shape[1]),
        "Missing_Required_Columns": ", ".join(missing_required) if missing_required else "",
        "Conc_Missing_Before_FFill": conc_missing_before,
        "Conc_Missing_After_FFill": conc_missing_after,
        "Control_Replicates": control_reps,
        "Treated_Levels": treated_levels,
        "Treated_Concentrations_ppm": ", ".join([str(x) for x in treated_conc_list]),
        "Other_Numeric_Missing_Cells": numeric_missing,
        "Structure_OK": len(missing_required) == 0
    })

    cleaned_preview[normalized_name] = df.head(8).copy()

audit_df = pd.DataFrame(audit_rows)

# ---------- 0.12 Display audit ----------
display(Markdown("## Step 0 workbook audit"))
display(audit_df)

for sname, preview_df in cleaned_preview.items():
    display(Markdown(f"### Preview: {sname}"))
    display(preview_df)

# ---------- 0.13 Workbook-level validation ----------
if not audit_df["Structure_OK"].all():
    bad_sheets = audit_df.loc[~audit_df["Structure_OK"], "Raw_Sheet_Name"].tolist()
    raise ValueError(
        "Workbook structure mismatch in these sheet(s): " + ", ".join(bad_sheets)
    )

# ---------- 0.14 Save audit outputs ----------
with pd.ExcelWriter(TARGET_AUDIT_XLSX, engine="openpyxl") as writer:
    audit_df.to_excel(writer, sheet_name="audit_summary", index=False)
    for sname, preview_df in cleaned_preview.items():
        safe_name = sname[:31]
        preview_df.to_excel(writer, sheet_name=safe_name, index=False)

project_metadata = {
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "target_workbook": str(TARGET_XLSX),
    "github_notebook_url": github_url,
    "github_url_source": github_source_file,
    "sheet_names_raw": sheet_names,
    "sheet_names_standardized": [normalize_sheet_name(s) for s in sheet_names],
    "analysis_template": ANALYSIS_TEMPLATE,
    "required_columns": required_columns
}

with open(TARGET_META_JSON, "w", encoding="utf-8") as f:
    json.dump(project_metadata, f, indent=2, ensure_ascii=False)

# ---------- 0.15 Verify metadata JSON ----------
with open(TARGET_META_JSON, "r", encoding="utf-8") as f:
    meta_check = json.load(f)

print("\nMetadata JSON verification:")
print("github_notebook_url =", meta_check.get("github_notebook_url"))
print("github_url_source   =", meta_check.get("github_url_source"))

# ---------- 0.16 Final console summary ----------
print("\n" + "="*70)
print("STEP 0 COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Workbook used           : {TARGET_XLSX}")
print(f"Audit file saved        : {TARGET_AUDIT_XLSX}")
print(f"Metadata file saved     : {TARGET_META_JSON}")
print(f"GitHub notebook URL     : {github_url}")
print(f"GitHub URL source       : {github_source_file}")

print("\nKey findings:")
print("- Workbook contains 6 insecticide sheets.")
print("- Real time points are 24, 48, and 72 HAS.")
print("- Ravzoom sheet has blank concentration cells that are fixed by forward-fill.")
print("- Metadata JSON now stores the GitHub notebook URL explicitly.")
print("- The dataset is ready for Step 1 preprocessing and Abbott correction.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root : /content/drive/MyDrive/Data Analysis March 2026
Output folder: /content/drive/MyDrive/Data Analysis March 2026/Mishu

Candidate Excel files found:
 - /content/drive/MyDrive/Data Analysis March 2026/Aleurodicus_rugioperculatus.xlsx
 - /content/drive/MyDrive/Data Analysis March 2026/Mishu/Aleurodicus_rugioperculatus.xlsx
 - /content/drive/MyDrive/CORRELATION AND P VALUE.xlsx
 - /content/drive/MyDrive/New Microsoft Excel Worksheet.xlsx
 - /content/drive/MyDrive/temporary upload/to_run.xlsx
 - /content/drive/MyDrive/temporary upload/Akib Main Data.xlsx
 - /content/drive/MyDrive/temporary upload/NILA Main Data.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Himel/himel.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Sadia/sadia.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Shormi/shormi.xlsx
 - /content/dr

## Step 0 workbook audit

,Raw_Sheet_Name,Standardized_Sheet_Name,Rows,Columns,Missing_Required_Columns,Conc_Missing_Before_FFill,Conc_Missing_After_FFill,Control_Replicates,Treated_Levels,Treated_Concentrations_ppm,Other_Numeric_Missing_Cells,Structure_OK
0,Tundra 20SP,Tundra 20 SP,28,16,,0,0,4,6,"2.0, 4.0, 8.0, 16.0, 32.0, 64.0",0,True
1,Actara 25 WG,Actara 25 WG,28,16,,0,0,4,6,"1.25, 2.5, 5.0, 10.0, 20.0, 40.0",0,True
2,Tiddo 20 SL,Tiddo 20 SL,28,16,,0,0,4,6,"1.0, 2.0, 4.0, 8.0, 16.0, 32.0",0,True
3,Shobicron 425 EC,Shobicron 425 EC,28,16,,0,0,4,6,"8.5, 17.0, 34.0, 68.0, 136.0, 272.0",0,True
4,Panel 50 SP,Panel 50 SP,28,16,,0,0,4,6,"5.0, 10.0, 20.0, 40.0, 80.0, 160.0",0,True
5,Ravzoom 14.5 SC,Rav-zoom 14.5 SC,28,16,,21,0,4,6,"1.015, 2.03, 4.06, 8.12, 16.24, 32.48",0,True


### Preview: Tundra 20 SP

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,58,55,50,44,54,51,45,39,12,10,7,5,control,0.0
1,control,R2,49,45,40,34,86,82,77,71,14,12,9,5,control,0.0
2,control,R3,76,72,67,61,75,71,65,58,8,7,5,3,control,0.0
3,control,R4,91,88,83,77,65,61,56,50,9,6,3,1,control,0.0
4,2ppm,R1,89,53,28,4,72,40,20,4,13,8,3,0,treated,2.0
5,2ppm,R2,72,60,34,18,71,59,43,26,15,12,7,0,treated,2.0
6,2ppm,R3,76,64,48,31,59,47,32,15,8,5,1,0,treated,2.0
7,2ppm,R4,58,47,33,17,57,45,30,13,9,6,2,0,treated,2.0


...

### Preview: Panel 50 SP

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,57,54,50,44,53,50,46,41,12,10,9,7,control,0.0
1,control,R2,28,25,20,14,43,41,36,30,10,9,9,8,control,0.0
2,control,R3,75,72,68,61,91,88,84,78,7,5,4,4,control,0.0
3,control,R4,38,35,31,25,30,27,23,16,9,7,5,3,control,0.0
4,5ppm,R1,68,48,41,28,78,49,44,39,15,13,10,6,treated,5.0
5,5ppm,R2,76,73,69,62,69,64,58,51,16,10,5,2,treated,5.0
6,5ppm,R3,75,71,65,59,56,52,47,40,14,9,5,1,treated,5.0
7,5ppm,R4,60,56,51,45,91,85,78,70,12,7,3,0,treated,5.0


### Preview: Rav-zoom 14.5 SC

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,59,56,52,47,54,50,45,39,13,11,8,5,control,0.000
1,control,R2,30,27,23,19,48,45,39,33,8,7,5,3,control,0.000
2,control,R3,76,72,68,63,92,88,83,76,11,9,6,3,control,0.000
3,control,R4,42,38,33,28,33,29,24,19,9,6,3,2,control,0.000
4,1.015ppm,R1,51,41,36,29,61,35,33,30,14,10,9,9,treated,1.015
5,1.015ppm,R2,48,46,42,37,57,53,48,41,15,8,5,2,treated,1.015
6,1.015ppm,R3,65,62,58,52,79,73,66,58,16,7,3,1,treated,1.015
7,1.015ppm,R4,44,42,39,34,75,69,62,55,13,6,3,1,treated,1.015



Metadata JSON verification:
github_notebook_url = https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb
github_url_source   = hardcoded_fallback

STEP 0 COMPLETED SUCCESSFULLY
Workbook used           : /content/drive/MyDrive/Data Analysis March 2026/Aleurodicus_rugioperculatus.xlsx
Audit file saved        : /content/drive/MyDrive/Data Analysis March 2026/Mishu/Step0_Workbook_Audit.xlsx
Metadata file saved     : /content/drive/MyDrive/Data Analysis March 2026/Mishu/Step0_Project_Metadata.json
GitHub notebook URL     : https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb
GitHub URL source       : hardcoded_fallback

Key findings:
- Workbook contains 6 insecticide sheets.
- Real time points are 24, 48, and 72 HAS.
- Ravzoom sheet has blank concentration cells that are fixed by forward-fill.
- Metadata JSON now stores the GitHub notebook URL explicitly.
- The dataset is ready for Step 